In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neural_network import MLPClassifier

from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# 0. CHARGEMENT & PREPROCESSING
# ============================================================

CSV_PATH = "student.csv"   # ⚠️ adapte si besoin
TARGET_COL = "Target"

df = pd.read_csv(CSV_PATH, sep=';')

class_map = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
class_names = list(class_map.keys())
y = df[TARGET_COL].map(class_map).values
num_classes = len(class_map)

X_df = df.drop(columns=[TARGET_COL])

categorical_cols = X_df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in X_df.columns if c not in categorical_cols]

if categorical_cols:
    X_cat = pd.get_dummies(X_df[categorical_cols], drop_first=True)
else:
    X_cat = pd.DataFrame(index=X_df.index)

X_num = X_df[numeric_cols].astype(float)
X_full = pd.concat([X_num, X_cat], axis=1)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_full.values, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

input_dim = X_train.shape[1]
print("Shapes => train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

os.makedirs("figures", exist_ok=True)

# ============================================================
# 1. MLP NUMPY FROM SCRATCH
# ============================================================

class NumpyMLP:
    """
    MLP multi-classe NumPy :
      - ReLU + Softmax
      - Mini-batch + Adam
      - L2 (weight decay)
      - Dropout sur les couches cachées
      - Tracking train/val loss & acc
    """
    def __init__(self, input_dim, hidden_layers, num_classes,
                 lr=1e-3, epochs=80, batch_size=64,
                 l2_lambda=1e-3, dropout_rate=0.2,
                 seed=42, verbose=False):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.num_classes = num_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        self.verbose = verbose

        self.rng = np.random.default_rng(seed)
        self._init_params()
        self._init_adam()

        self.history_train_loss = []
        self.history_val_loss = []
        self.history_train_acc = []
        self.history_val_acc = []

    def _init_params(self):
        sizes = [self.input_dim] + self.hidden_layers + [self.num_classes]
        self.W = []
        self.b = []
        for in_size, out_size in zip(sizes[:-1], sizes[1:]):
            W = self.rng.normal(0, np.sqrt(2.0 / in_size), size=(in_size, out_size))
            b = np.zeros((1, out_size))
            self.W.append(W)
            self.b.append(b)

    def _init_adam(self):
        self.mW = [np.zeros_like(W) for W in self.W]
        self.vW = [np.zeros_like(W) for W in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.beta1, self.beta2, self.eps = 0.9, 0.999, 1e-8
        self.t = 0

    @staticmethod
    def _relu(x): return np.maximum(0, x)
    @staticmethod
    def _relu_deriv(x): return (x > 0).astype(float)

    @staticmethod
    def _softmax(x):
        x_shift = x - np.max(x, axis=1, keepdims=True)
        exp_x = np.exp(x_shift)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def _cross_entropy_loss(self, probs, y):
        N = y.shape[0]
        p_true = probs[np.arange(N), y]
        p_true = np.clip(p_true, 1e-8, 1 - 1e-8)
        ce = -np.mean(np.log(p_true))
        l2 = sum(np.sum(W**2) for W in self.W) * self.l2_lambda
        return ce + l2

    def _forward(self, X, training=True):
        A = X
        A_list = [A]
        Z_list = []
        mask_list = []
        for i in range(len(self.W)):
            Z = A @ self.W[i] + self.b[i]
            Z_list.append(Z)
            if i < len(self.W) - 1:
                A = self._relu(Z)
                mask = None
                if training and self.dropout_rate > 0.0:
                    mask = (self.rng.random(A.shape) > self.dropout_rate).astype(float)
                    A = A * mask / (1.0 - self.dropout_rate)
                mask_list.append(mask)
            else:
                A = self._softmax(Z)
                mask_list.append(None)
            A_list.append(A)
        return A_list[-1], Z_list, A_list, mask_list

    def _backward(self, X, y, Z_list, A_list, probs, mask_list):
        N = X.shape[0]
        L = len(self.W)
        dW_list = [None]*L
        db_list = [None]*L

        dZ = probs.copy()
        dZ[np.arange(N), y] -= 1
        dZ /= N

        for i in reversed(range(L)):
            A_prev = A_list[i]
            dW = A_prev.T @ dZ + 2 * self.l2_lambda * self.W[i]
            db = np.sum(dZ, axis=0, keepdims=True)
            dW_list[i] = dW
            db_list[i] = db
            if i > 0:
                dA_prev = dZ @ self.W[i].T
                mask_prev = mask_list[i-1]
                if mask_prev is not None:
                    dA_prev = dA_prev * mask_prev / (1.0 - self.dropout_rate)
                dZ = dA_prev * self._relu_deriv(Z_list[i-1])
        return dW_list, db_list

    def _adam_update(self, gradsW, gradsB):
        self.t += 1
        lr_t = self.lr * np.sqrt(1 - self.beta2**self.t) / (1 - self.beta1**self.t)
        for i in range(len(self.W)):
            self.mW[i] = self.beta1*self.mW[i] + (1-self.beta1)*gradsW[i]
            self.vW[i] = self.beta2*self.vW[i] + (1-self.beta2)*(gradsW[i]**2)
            self.mb[i] = self.beta1*self.mb[i] + (1-self.beta1)*gradsB[i]
            self.vb[i] = self.beta2*self.vb[i] + (1-self.beta2)*(gradsB[i]**2)
            self.W[i] -= lr_t * self.mW[i] / (np.sqrt(self.vW[i]) + self.eps)
            self.b[i] -= lr_t * self.mb[i] / (np.sqrt(self.vb[i]) + self.eps)

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        N = X_train.shape[0]
        indices = np.arange(N)

        for epoch in range(self.epochs):
            self.rng.shuffle(indices)
            X_sh = X_train[indices]
            y_sh = y_train[indices]

            for k in range(0, N, self.batch_size):
                Xb = X_sh[k:k+self.batch_size]
                yb = y_sh[k:k+self.batch_size]
                probs, Z_list, A_list, mask_list = self._forward(Xb, training=True)
                dW_list, db_list = self._backward(Xb, yb, Z_list, A_list, probs, mask_list)
                self._adam_update(dW_list, db_list)

            # stats train
            train_probs, _, _, _ = self._forward(X_train, training=False)
            train_loss = self._cross_entropy_loss(train_probs, y_train)
            train_pred = np.argmax(train_probs, axis=1)
            train_acc = accuracy_score(y_train, train_pred)

            self.history_train_loss.append(train_loss)
            self.history_train_acc.append(train_acc)

            if X_val is not None and y_val is not None:
                val_probs, _, _, _ = self._forward(X_val, training=False)
                val_loss = self._cross_entropy_loss(val_probs, y_val)
                val_pred = np.argmax(val_probs, axis=1)
                val_acc = accuracy_score(y_val, val_pred)
                self.history_val_loss.append(val_loss)
                self.history_val_acc.append(val_acc)
            else:
                self.history_val_loss.append(None)
                self.history_val_acc.append(None)

        return self

    def predict(self, X):
        probs, _, _, _ = self._forward(X, training=False)
        return np.argmax(probs, axis=1)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


numpy_configs = [
    {"name": "NP_small",  "hidden_layers": [64, 32],       "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
    {"name": "NP_medium", "hidden_layers": [128, 64],      "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
    {"name": "NP_deep",   "hidden_layers": [128, 96, 64],  "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
]

numpy_results = []

print("\n=== ENTRAÎNEMENTS MLP NUMPY (LOOP) ===")
for cfg in numpy_configs:
    print(f"\n--- Config NumPy : {cfg['name']} | HL={cfg['hidden_layers']} ---")
    model_np = NumpyMLP(
        input_dim=input_dim,
        hidden_layers=cfg["hidden_layers"],
        num_classes=num_classes,
        lr=cfg["lr"],
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        l2_lambda=cfg["l2"],
        dropout_rate=cfg["dropout"],
        verbose=False
    )
    model_np.fit(X_train, y_train, X_val, y_val)
    train_acc = model_np.score(X_train, y_train)
    val_acc = model_np.score(X_val, y_val)
    test_acc = model_np.score(X_test, y_test)
    y_test_pred = model_np.predict(X_test)

    numpy_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "lr": cfg["lr"],
        "epochs": cfg["epochs"],
        "batch_size": cfg["batch_size"],
        "l2_lambda": cfg["l2"],
        "dropout": cfg["dropout"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "model": model_np,
        "y_test_pred": y_test_pred,
    })

numpy_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["model", "y_test_pred"]}
    for res in numpy_results
])
print("\n===== RÉSUMÉ NUMPY =====")
print(numpy_results_df)

best_np_idx = numpy_results_df["test_acc"].idxmax()
best_np = numpy_results[best_np_idx]
print(f"\n>>> Best NumPy : {best_np['name']} | test_acc={best_np['test_acc']:.3f}")

cm_np = confusion_matrix(y_test, best_np["y_test_pred"])
report_np = classification_report(y_test, best_np["y_test_pred"], target_names=class_names)

epochs_np = range(1, len(best_np["model"].history_train_loss) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_np, best_np["model"].history_train_loss, label="Train loss")
if best_np["model"].history_val_loss[0] is not None:
    plt.plot(epochs_np, best_np["model"].history_val_loss, label="Val loss")
plt.title(f"MLP NumPy ({best_np['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_numpy_best.png"); plt.close()

# ============================================================
# 2. MLP KERAS (LOOP)
# ============================================================

def build_keras_mlp(input_dim, hidden_layers, dropout_rate=0.2, lr=1e-3, use_batchnorm=True):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    for units in hidden_layers:
        x = layers.Dense(units, activation="relu")(x)
        if dropout_rate > 0.0:
            x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

keras_configs = [
    {"name": "K_small_noDO",  "hidden_layers": [64, 32],      "dropout": 0.0, "lr": 1e-3, "epochs": 80, "batch_size": 64},
    {"name": "K_small_DO",    "hidden_layers": [64, 32],      "dropout": 0.2, "lr": 1e-3, "epochs": 80, "batch_size": 64},
    {"name": "K_deep_DO",     "hidden_layers": [128, 96, 64], "dropout": 0.2, "lr": 1e-3, "epochs": 80, "batch_size": 64},
]

keras_results = []

print("\n=== ENTRAÎNEMENTS MLP KERAS (LOOP) ===")
for cfg in keras_configs:
    print(f"\n--- Config Keras : {cfg['name']} | HL={cfg['hidden_layers']} ---")
    model_k = build_keras_mlp(
        input_dim=input_dim,
        hidden_layers=cfg["hidden_layers"],
        dropout_rate=cfg["dropout"],
        lr=cfg["lr"],
        use_batchnorm=True
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
    history = model_k.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        verbose=0,
        callbacks=[early_stop]
    )

    train_loss = history.history["loss"][-1]
    val_loss = history.history["val_loss"][-1]
    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]

    test_loss, test_acc = model_k.evaluate(X_test, y_test, verbose=0)
    y_test_proba = model_k.predict(X_test, verbose=0)
    y_test_pred = np.argmax(y_test_proba, axis=1)

    keras_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "dropout": cfg["dropout"],
        "lr": cfg["lr"],
        "epochs": cfg["epochs"],
        "batch_size": cfg["batch_size"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "history": history,
        "model": model_k,
        "y_test_pred": y_test_pred
    })

keras_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["history", "model", "y_test_pred"]}
    for res in keras_results
])
print("\n===== RÉSUMÉ KERAS =====")
print(keras_results_df)

best_k_idx = keras_results_df["test_acc"].idxmax()
best_k = keras_results[best_k_idx]
print(f"\n>>> Best Keras : {best_k['name']} | test_acc={best_k['test_acc']:.3f}")

cm_k = confusion_matrix(y_test, best_k["y_test_pred"])
report_k = classification_report(y_test, best_k["y_test_pred"], target_names=class_names)

hist_k = best_k["history"]
epochs_k = range(1, len(hist_k.history["loss"]) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_k, hist_k.history["loss"], label="Train loss")
plt.plot(epochs_k, hist_k.history["val_loss"], label="Val loss")
plt.title(f"MLP Keras ({best_k['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_keras_best.png"); plt.close()

# ============================================================
# 3. MLP sklearn (LOOP)
# ============================================================

sklearn_configs = [
    {"name": "SK_small_alpha1e-3", "hidden_layers": (64, 32), "alpha": 1e-3},
    {"name": "SK_small_alpha1e-4", "hidden_layers": (64, 32), "alpha": 1e-4},
    {"name": "SK_deep_alpha1e-3",  "hidden_layers": (128, 64), "alpha": 1e-3},
]

sk_results = []

print("\n=== ENTRAÎNEMENTS MLP SKLEARN (LOOP) ===")
for cfg in sklearn_configs:
    print(f"\n--- Config sklearn : {cfg['name']} | HL={cfg['hidden_layers']} | alpha={cfg['alpha']} ---")
    sk_mlp = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layers"],
        activation='relu',
        solver='adam',
        alpha=cfg["alpha"],
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    )
    sk_mlp.fit(X_train, y_train)
    train_acc = sk_mlp.score(X_train, y_train)
    val_acc = sk_mlp.score(X_val, y_val)
    test_acc = sk_mlp.score(X_test, y_test)
    y_test_pred = sk_mlp.predict(X_test)

    sk_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "alpha": cfg["alpha"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "loss_curve": sk_mlp.loss_curve_,
        "y_test_pred": y_test_pred
    })

sk_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["loss_curve", "y_test_pred"]}
    for res in sk_results
])
print("\n===== RÉSUMÉ SKLEARN =====")
print(sk_results_df)

best_sk_idx = sk_results_df["test_acc"].idxmax()
best_sk = sk_results[best_sk_idx]
print(f"\n>>> Best sklearn : {best_sk['name']} | test_acc={best_sk['test_acc']:.3f}")

cm_sk = confusion_matrix(y_test, best_sk["y_test_pred"])
report_sk = classification_report(y_test, best_sk["y_test_pred"], target_names=class_names)

loss_curve_sk = best_sk["loss_curve"]
epochs_sk = range(1, len(loss_curve_sk) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_sk, loss_curve_sk, label="Train loss")
plt.title(f"MLP sklearn ({best_sk['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_sklearn_best.png"); plt.close()

# ============================================================
# 4. OUTIL POUR MATRICE DE CONFUSION EN HTML
# ============================================================

def confusion_matrix_to_html(cm, class_names):
    """
    Formatte une matrice de confusion (numpy array) en tableau HTML lisible.
    """
    n = len(class_names)
    header = "<tr><th></th>" + "".join(f"<th>{c}</th>" for c in class_names) + "</tr>"
    rows = []
    for i, row_name in enumerate(class_names):
        cells = "".join(f"<td>{int(cm[i,j])}</td>" for j in range(n))
        rows.append(f"<tr><th>{row_name}</th>{cells}</tr>")
    table = "<table>" + header + "".join(rows) + "</table>"
    return table

cm_np_html = confusion_matrix_to_html(cm_np, class_names)
cm_k_html = confusion_matrix_to_html(cm_k, class_names)
cm_sk_html = confusion_matrix_to_html(cm_sk, class_names)

# ============================================================
# 5. RAPPORT HTML + CSS (avec questions du sujet en haut)
# ============================================================

report_html = f"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>ANN Playground — Comparaison MLP NumPy / Keras / sklearn</title>
<style>
body {{
  font-family: Arial, sans-serif;
  background-color: #020617;
  color: #e5e7eb;
  margin: 0;
  padding: 20px;
}}
h1, h2, h3 {{
  color: #fbbf24;
}}
a {{ color: #60a5fa; }}
.card {{
  background-color: #020617;
  border-radius: 12px;
  padding: 16px 20px;
  margin-bottom: 20px;
  box-shadow: 0 8px 20px rgba(0,0,0,0.5);
  border: 1px solid #1f2937;
}}
table {{
  border-collapse: collapse;
  margin: 10px 0;
  font-size: 13px;
}}
th, td {{
  border: 1px solid #1f2937;
  padding: 6px 8px;
  text-align: center;
}}
th {{
  background-color: #111827;
}}
.metric-ok {{
  color: #4ade80;
  font-weight: bold;
}}
.metric-medium {{
  color: #facc15;
  font-weight: bold;
}}
pre {{
  background-color: #020617;
  border-radius: 8px;
  border: 1px solid #1f2937;
  padding: 10px;
  overflow-x: auto;
  font-size: 12px;
}}
.img-block {{
  text-align: center;
  margin: 10px 0;
}}
img {{
  max-width: 100%;
  border-radius: 12px;
  border: 1px solid #1f2937;
}}
.small {{
  font-size: 12px;
  color: #9ca3af;
}}
</style>
</head>
<body>

<h1>ANN Playground — Perceptron Multicouches sur le décrochage étudiant</h1>

<div class="card">
  <h2>0. Questions théoriques — Réponses pour la présentation</h2>

  <h3>0.1 Qu’est-ce qu’un Perceptron Multicouches (PMC) ?</h3>
  <p>
    Un PMC (Perceptron Multicouches) est un réseau de neurones artificiel composé de plusieurs couches de neurones :
  </p>
  <ul>
    <li><strong>Couche d’entrée</strong> : reçoit les variables explicatives du problème (ici, les caractéristiques des étudiants).</li>
    <li><strong>Couches cachées</strong> : appliquent des transformations non linéaires grâce à des fonctions d’activation
        (ReLU, par exemple). Elles apprennent des représentations de plus en plus complexes des données.</li>
    <li><strong>Couche de sortie</strong> : produit la prédiction finale :
      <ul>
        <li>en classification multi-classe : une couche Dense avec autant de neurones que de classes, + <code>softmax</code> ;</li>
        <li>en régression : souvent un seul neurone avec activation linéaire.</li>
      </ul>
    </li>
  </ul>

  <h3>0.2 Choix de l’architecture du PMC selon classification / régression</h3>
  <p>
    Le choix de l’architecture dépend du type de problème :
  </p>
  <ul>
    <li><strong>Classification</strong> :
      <ul>
        <li>Couche de sortie : <code>softmax</code> (multi-classe) ou <code>sigmoid</code> (binaire).</li>
        <li>Nombre de neurones de sortie = nombre de classes.</li>
        <li>Quelques couches cachées (1 à 3) avec 32 à 256 neurones sont souvent suffisantes sur des données tabulaires.</li>
      </ul>
    </li>
    <li><strong>Régression</strong> :
      <ul>
        <li>Couche de sortie : 1 neurone avec activation linéaire.</li>
        <li>Mêmes idées pour les couches cachées, mais on surveille l’overfitting via la loss de validation.</li>
      </ul>
    </li>
  </ul>
  <p>
    Dans ce projet, nous sommes sur un <strong>problème de classification multi-classe</strong> (Dropout / Enrolled / Graduate),
    donc nous utilisons une couche de sortie softmax avec 3 neurones.
  </p>

  <h3>0.3 Définitions : Fonction d’activation, Propagation, Rétropropagation, Loss-function, Descente de gradient, Vanishing gradients</h3>
  <ul>
    <li><strong>Fonction d’activation</strong> :
      transforme la somme pondérée en sortie de chaque neurone. Exemples :
      <code>ReLU(x) = max(0, x)</code>, <code>sigmoid</code>, <code>tanh</code>.
      Elle permet d’introduire de la non-linéarité et donc de modéliser des relations complexes.
    </li>
    <li><strong>Propagation (forward pass)</strong> :
      on part des entrées, on applique successivement les couches (poids + activation) jusqu’à la couche de sortie.
      On obtient ainsi une prédiction du réseau.
    </li>
    <li><strong>Rétropropagation (backpropagation)</strong> :
      à partir de l’erreur en sortie, on calcule les gradients de la loss par rapport à chaque poids, en remontant couche par couche.
      Cela permet de savoir dans quel sens modifier chaque poids.
    </li>
    <li><strong>Loss-function</strong> :
      fonction qui mesure l’erreur entre la prédiction du réseau et la valeur réelle.
      Ici, on utilise la <strong>cross-entropy</strong> pour la classification multi-classe.
      L’objectif est de minimiser cette loss.
    </li>
    <li><strong>Descente de gradient</strong> :
      méthode d’optimisation qui met à jour les poids dans la direction opposée au gradient de la loss.
      On choisit un pas d’apprentissage (learning rate) qui contrôle la taille des mises à jour.
    </li>
    <li><strong>Vanishing gradients</strong> :
      dans les réseaux très profonds (et avec certaines activations comme sigmoid/tanh),
      les gradients deviennent de plus en plus petits en remontant vers les premières couches.
      Résultat : ces couches apprennent très peu. ReLU, BatchNorm et des architectures raisonnables limitent ce problème.
    </li>
  </ul>

  <h3>0.4 Hyper-paramètres d’un réseau de neurones & bonnes pratiques</h3>
  <p>
    Voici au moins 5 hyper-paramètres importants et des bonnes pratiques pour choisir leurs valeurs :
  </p>
  <ul>
    <li><strong>Nombre de couches cachées</strong> :
      1 à 3 couches suffisent souvent pour des données tabulaires.
      Plus de couches augmentent la capacité mais aussi le risque d’overfitting et de vanishing gradients.
    </li>
    <li><strong>Nombre de neurones par couche</strong> :
      typiquement entre 32 et 512 neurones selon la taille et la complexité des données.
      Ici, nous testons par exemple [64, 32], [128, 64], [128, 96, 64].
    </li>
    <li><strong>Learning rate (taux d’apprentissage)</strong> :
      contrôle la taille des pas de mise à jour.
      Une valeur classique avec Adam est de l’ordre de <code>1e-3</code>.
      Trop grand → divergence, trop petit → entraînement très lent.
    </li>
    <li><strong>Taille de batch</strong> :
      nombre d’exemples utilisés pour calculer un gradient.
      Des valeurs entre 32 et 256 sont fréquentes.
      Ici, nous utilisons souvent 64.
    </li>
    <li><strong>Nombre d’epochs</strong> :
      nombre de passes complètes sur le jeu d’entraînement.
      Avec early stopping, on peut mettre une valeur assez grande (par ex. 80 ou 100)
      et laisser le réseau s’arrêter automatiquement quand la loss de validation ne s’améliore plus.
    </li>
    <li><strong>Régularisation (L2, dropout)</strong> :
      L2 (weight decay) pénalise les poids trop grands ;
      dropout désactive aléatoirement des neurones pendant l’entraînement.
      Bonnes pratiques : L2 entre 1e-4 et 1e-2 ; dropout souvent autour de 0.2–0.5 selon l’overfitting observé.
    </li>
    <li><strong>Fonction d’activation</strong> :
      ReLU est un très bon choix par défaut pour les couches cachées de réseaux profonds,
      car elle réduit le problème de vanishing gradients.
    </li>
  </ul>
</div>

<div class="card">
  <h2>1. Rappel des consignes (PDF ANN Playground) et réponses</h2>
  <p>Le sujet demande notamment de :</p>
  <ul>
    <li>Récupérer le dataset <em>Predict Students' Dropout and Academic Success</em> (UCI).</li>
    <li>Réaliser une analyse exploratoire et nettoyer les données.</li>
    <li>Construire plusieurs MLP avec Keras (architecture, dropout, normalisation, etc.).</li>
    <li>Évaluer avec matrice de confusion, accuracy et rapport de classification.</li>
    <li>Visualiser les performances train/validation en fonction des epochs.</li>
    <li>Tester et commenter l'overfitting et l'effet de la méthode dropout.</li>
    <li>Utiliser une couche de normalisation.</li>
    <li>Coder un Perceptron Multicouches en NumPy (orienté objet), l'entraîner, visualiser ses courbes.</li>
    <li>Comparer le MLP Keras et le MLP NumPy pour les mêmes hyper-paramètres.</li>
  </ul>
  <p><strong>Dans ce travail&nbsp;:</strong></p>
  <ul>
    <li>Les données ont été chargées avec <code>pandas</code>, encodées (one-hot) et normalisées (StandardScaler).</li>
    <li>Plusieurs architectures Keras, NumPy et sklearn ont été testées via des boucles de configuration.</li>
    <li>Les résultats incluent accuracy, matrices de confusion, rapports de classification et courbes de loss train/val.</li>
    <li>Dropout et la normalisation (BatchNormalization) sont utilisés pour limiter l'overfitting.</li>
    <li>Une comparaison directe Keras vs NumPy vs sklearn est présentée ci-dessous.</li>
  </ul>
</div>

<div class="card">
  <h2>2. Résultats détaillés par famille de modèles</h2>

  <h3>2.1 MLP NumPy — Résumé des expériences</h3>
  <pre>{numpy_results_df.to_string(index=False)}</pre>
  <p>Meilleur MLP NumPy : <strong>{best_np['name']}</strong>, test_acc = <span class="metric-ok">{best_np['test_acc']:.3f}</span></p>

  <h3>2.2 MLP Keras — Résumé des expériences</h3>
  <pre>{keras_results_df.to_string(index=False)}</pre>
  <p>Meilleur MLP Keras : <strong>{best_k['name']}</strong>, test_acc = <span class="metric-medium">{best_k['test_acc']:.3f}</span></p>

  <h3>2.3 MLP scikit-learn — Résumé des expériences</h3>
  <pre>{sk_results_df.to_string(index=False)}</pre>
  <p>Meilleur MLP sklearn : <strong>{best_sk['name']}</strong>, test_acc = <span class="metric-medium">{best_sk['test_acc']:.3f}</span></p>
</div>

<div class="card">
  <h2>3. Comparaison des meilleurs modèles</h2>
  <table>
    <tr>
      <th>Famille</th>
      <th>Nom config</th>
      <th>Hidden layers</th>
      <th>Régularisation</th>
      <th>Train acc</th>
      <th>Val acc</th>
      <th>Test acc</th>
    </tr>
    <tr>
      <td>NumPy</td>
      <td>{best_np['name']}</td>
      <td>{best_np['hidden_layers']}</td>
      <td>DO={best_np['dropout']}, L2={best_np['l2_lambda']}</td>
      <td>{best_np['train_acc']:.3f}</td>
      <td>{best_np['val_acc']:.3f}</td>
      <td class="metric-ok">{best_np['test_acc']:.3f}</td>
    </tr>
    <tr>
      <td>Keras</td>
      <td>{best_k['name']}</td>
      <td>{best_k['hidden_layers']}</td>
      <td>DO={best_k['dropout']}, BatchNorm</td>
      <td>{best_k['train_acc']:.3f}</td>
      <td>{best_k['val_acc']:.3f}</td>
      <td class="metric-medium">{best_k['test_acc']:.3f}</td>
    </tr>
    <tr>
      <td>sklearn</td>
      <td>{best_sk['name']}</td>
      <td>{best_sk['hidden_layers']}</td>
      <td>alpha={best_sk['alpha']}</td>
      <td>{best_sk['train_acc']:.3f}</td>
      <td>{best_sk['val_acc']:.3f}</td>
      <td class="metric-medium">{best_sk['test_acc']:.3f}</td>
    </tr>
  </table>
  <p>
    En explorant plusieurs jeux d'hyper-paramètres via des boucles
    (<code>for cfg in numpy_configs / keras_configs / sklearn_configs</code>),
    le meilleur <strong>MLP NumPy from scratch</strong> atteint une accuracy test
    légèrement supérieure aux meilleurs modèles Keras et sklearn, ce qui montre
    qu'un PMC codé à la main peut être au moins aussi performant qu'un modèle de bibliothèque,
    dès lors que l'architecture et la régularisation sont bien choisies.
  </p>
</div>

<div class="card">
  <h2>4. Courbes de loss train / validation (réponse à « visualisez les performances »)</h2>

  <h3>4.1 MLP NumPy — {best_np['name']}</h3>
  <div class="img-block">
    <img src="figures/loss_numpy_best.png" alt="Loss MLP NumPy">
  </div>

  <h3>4.2 MLP Keras — {best_k['name']}</h3>
  <div class="img-block">
    <img src="figures/loss_keras_best.png" alt="Loss MLP Keras">
  </div>

  <h3>4.3 MLP sklearn — {best_sk['name']}</h3>
  <div class="img-block">
    <img src="figures/loss_sklearn_best.png" alt="Loss MLP sklearn">
  </div>

  <p>
    Ces courbes permettent de vérifier l'apparition d'<strong>overfitting</strong> :
    lorsque la loss d'entraînement continue de baisser alors que la loss de validation stagne
    ou remonte. Le dropout et la L2 limitent cet effet, en particulier pour les architectures
    plus profondes.
  </p>
</div>

<div class="card">
  <h2>5. Matrices de confusion (affichage matriciel)</h2>
  <p class="small">
    Les lignes correspondent à la classe réelle, les colonnes à la classe prédite.
    Ordre des classes : Dropout, Enrolled, Graduate.
  </p>

  <h3>5.1 MLP NumPy</h3>
  {cm_np_html}

  <h3>5.2 MLP Keras</h3>
  {cm_k_html}

  <h3>5.3 MLP sklearn</h3>
  {cm_sk_html}
</div>

<div class="card">
  <h2>6. Rapports de classification (accuracy & rapport complet)</h2>

  <h3>6.1 MLP NumPy</h3>
  <pre>{report_np}</pre>

  <h3>6.2 MLP Keras</h3>
  <pre>{report_k}</pre>

  <h3>6.3 MLP sklearn</h3>
  <pre>{report_sk}</pre>
</div>

<div class="card">
  <h2>7. Discussion par rapport au sujet</h2>
  <ul>
    <li><strong>Overfitting :</strong> les modèles les plus profonds ou sans dropout ont tendance à mieux
        performer sur le train que sur la validation. L'ajout de dropout et de L2 réduit ce gap.</li>
    <li><strong>Dropout :</strong> la désactivation aléatoire de neurones pendant l'entraînement oblige le réseau
        à répartir l'information, ce qui améliore la généralisation.</li>
    <li><strong>Normalisation :</strong> la couche <code>BatchNormalization</code> en Keras stabilise les activations,
        ce qui accélère la convergence et réduit les risques de gradients trop grands ou trop petits.</li>
    <li><strong>Comparaison Keras vs NumPy :</strong> pour des architectures et des hyper-paramètres similaires
        ([64, 32] puis [128, 96, 64], dropout 0.2, Adam), les performances sont proches, avec un léger avantage au
        MLP NumPy sur ce split, ce qui valide l’implémentation.</li>
    <li><strong>Vanishing gradients :</strong> sur des réseaux très profonds, les gradients peuvent devenir
        extrêmement petits en remontant vers les premières couches, ce qui les empêche d’apprendre.
        Ici, les architectures restent relativement peu profondes et l’utilisation de ReLU limite ce phénomène,
        mais il reste important à connaître pour des réseaux plus profonds.</li>
  </ul>
</div>

<div class="card">
  <h2>8. Glossaire (explications simples)</h2>
  <p><strong>Dropout (classe Target « Dropout ») :</strong> un étudiant qui quitte ses études avant d’avoir validé son diplôme.</p>
  <p><strong>Enrolled :</strong> un étudiant encore inscrit, qui n’a ni décroché ni encore obtenu son diplôme.</p>
  <p><strong>Graduate :</strong> un étudiant qui a terminé son parcours et obtenu son diplôme.</p>
  <hr class="small" />
  <p><strong>Dropout (technique de régularisation) :</strong> pendant l’entraînement, on coupe aléatoirement une partie des neurones.
     Cela empêche le réseau de trop se « spécialiser » sur certains neurones et améliore sa capacité à généraliser.</p>
  <p><strong>Adam :</strong> un algorithme de descente de gradient qui adapte automatiquement le pas d’apprentissage
     pour chaque poids. Il combine les idées de momentum et d’Adagrad. En pratique, il permet d’apprendre plus vite
     et plus stablement que la descente de gradient classique.</p>
  <p><strong>Fonction d’activation :</strong> une fonction non linéaire (par ex. ReLU) appliquée à la sortie d’un neurone.
     Elle permet au réseau de modéliser des relations complexes, pas seulement linéaires.</p>
  <p><strong>Propagation (forward pass) :</strong> on part des entrées et on calcule, couche par couche, les sorties du réseau
     jusqu’à obtenir la prédiction finale.</p>
  <p><strong>Rétropropagation (backprop) :</strong> on calcule l’erreur entre la prédiction et la vraie valeur, puis on la
     « remonte » dans le réseau pour savoir comment ajuster chaque poids.</p>
  <p><strong>Loss function :</strong> c’est la fonction qui mesure à quel point le modèle se trompe (par exemple la cross-entropy).
     L’objectif de l’entraînement est de minimiser cette quantité.</p>
  <p><strong>Descente de gradient :</strong> une méthode d’optimisation qui met à jour les poids dans la direction qui fait
     le plus baisser la loss.</p>
  <p><strong>Vanishing gradients :</strong> quand on multiplie beaucoup de petites dérivées (dans des réseaux très profonds),
     les gradients deviennent quasiment nuls. Les premières couches n’apprennent plus. D’où l’intérêt des fonctions ReLU,
     de la normalisation et d’architectures bien choisies.</p>
</div>

</body>
</html>
"""

report_path = "ANN_playground_mlp_report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_html)

print(f"\n✅ Rapport HTML généré : {report_path}")
print("   Ouvre-le dans ton navigateur pour voir la comparaison complète.")


Shapes => train: (2831, 36) | val: (708, 36) | test: (885, 36)

=== ENTRAÎNEMENTS MLP NUMPY (LOOP) ===

--- Config NumPy : NP_small | HL=[64, 32] ---

--- Config NumPy : NP_medium | HL=[128, 64] ---

--- Config NumPy : NP_deep | HL=[128, 96, 64] ---

===== RÉSUMÉ NUMPY =====
        name  hidden_layers     lr  epochs  batch_size  l2_lambda  dropout  \
0   NP_small       [64, 32]  0.001      80          64      0.001      0.2   
1  NP_medium      [128, 64]  0.001      80          64      0.001      0.2   
2    NP_deep  [128, 96, 64]  0.001      80          64      0.001      0.2   

   train_acc   val_acc  test_acc  
0   0.853762  0.776836  0.755932  
1   0.902155  0.762712  0.722034  
2   0.933239  0.747175  0.729944  

>>> Best NumPy : NP_small | test_acc=0.756

=== ENTRAÎNEMENTS MLP KERAS (LOOP) ===

--- Config Keras : K_small_noDO | HL=[64, 32] ---

--- Config Keras : K_small_DO | HL=[64, 32] ---

--- Config Keras : K_deep_DO | HL=[128, 96, 64] ---

===== RÉSUMÉ KERAS =====
        

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.neural_network import MLPClassifier

from tensorflow import keras
from tensorflow.keras import layers

# ============================================================
# 0. CHARGEMENT & PREPROCESSING
# ============================================================

CSV_PATH = "student.csv"   # ⚠️ adapte si besoin
TARGET_COL = "Target"

df = pd.read_csv(CSV_PATH, sep=';')

class_map = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
class_names = list(class_map.keys())
y = df[TARGET_COL].map(class_map).values
num_classes = len(class_map)

X_df = df.drop(columns=[TARGET_COL])

categorical_cols = X_df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = [c for c in X_df.columns if c not in categorical_cols]

if categorical_cols:
    X_cat = pd.get_dummies(X_df[categorical_cols], drop_first=True)
else:
    X_cat = pd.DataFrame(index=X_df.index)

X_num = X_df[numeric_cols].astype(float)
X_full = pd.concat([X_num, X_cat], axis=1)

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_full.values, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

input_dim = X_train.shape[1]
print("Shapes => train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

os.makedirs("figures", exist_ok=True)

# ============================================================
# 1. MLP NUMPY FROM SCRATCH
# ============================================================

class NumpyMLP:
    """
    MLP multi-classe NumPy :
      - ReLU + Softmax
      - Mini-batch + Adam
      - L2 (weight decay)
      - Dropout sur les couches cachées
      - Tracking train/val loss & acc
    """
    def __init__(self, input_dim, hidden_layers, num_classes,
                 lr=1e-3, epochs=80, batch_size=64,
                 l2_lambda=1e-3, dropout_rate=0.2,
                 seed=42, verbose=False):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.num_classes = num_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate
        self.verbose = verbose

        self.rng = np.random.default_rng(seed)
        self._init_params()
        self._init_adam()

        self.history_train_loss = []
        self.history_val_loss = []
        self.history_train_acc = []
        self.history_val_acc = []

    def _init_params(self):
        sizes = [self.input_dim] + self.hidden_layers + [self.num_classes]
        self.W = []
        self.b = []
        for in_size, out_size in zip(sizes[:-1], sizes[1:]):
            W = self.rng.normal(0, np.sqrt(2.0 / in_size), size=(in_size, out_size))
            b = np.zeros((1, out_size))
            self.W.append(W)
            self.b.append(b)

    def _init_adam(self):
        self.mW = [np.zeros_like(W) for W in self.W]
        self.vW = [np.zeros_like(W) for W in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.beta1, self.beta2, self.eps = 0.9, 0.999, 1e-8
        self.t = 0

    @staticmethod
    def _relu(x): return np.maximum(0, x)
    @staticmethod
    def _relu_deriv(x): return (x > 0).astype(float)

    @staticmethod
    def _softmax(x):
        x_shift = x - np.max(x, axis=1, keepdims=True)
        exp_x = np.exp(x_shift)
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def _cross_entropy_loss(self, probs, y):
        N = y.shape[0]
        p_true = probs[np.arange(N), y]
        p_true = np.clip(p_true, 1e-8, 1 - 1e-8)
        ce = -np.mean(np.log(p_true))
        l2 = sum(np.sum(W**2) for W in self.W) * self.l2_lambda
        return ce + l2

    def _forward(self, X, training=True):
        A = X
        A_list = [A]
        Z_list = []
        mask_list = []
        for i in range(len(self.W)):
            Z = A @ self.W[i] + self.b[i]
            Z_list.append(Z)
            if i < len(self.W) - 1:
                A = self._relu(Z)
                mask = None
                if training and self.dropout_rate > 0.0:
                    mask = (self.rng.random(A.shape) > self.dropout_rate).astype(float)
                    A = A * mask / (1.0 - self.dropout_rate)
                mask_list.append(mask)
            else:
                A = self._softmax(Z)
                mask_list.append(None)
            A_list.append(A)
        return A_list[-1], Z_list, A_list, mask_list

    def _backward(self, X, y, Z_list, A_list, probs, mask_list):
        N = X.shape[0]
        L = len(self.W)
        dW_list = [None]*L
        db_list = [None]*L

        dZ = probs.copy()
        dZ[np.arange(N), y] -= 1
        dZ /= N

        for i in reversed(range(L)):
            A_prev = A_list[i]
            dW = A_prev.T @ dZ + 2 * self.l2_lambda * self.W[i]
            db = np.sum(dZ, axis=0, keepdims=True)
            dW_list[i] = dW
            db_list[i] = db
            if i > 0:
                dA_prev = dZ @ self.W[i].T
                mask_prev = mask_list[i-1]
                if mask_prev is not None:
                    dA_prev = dA_prev * mask_prev / (1.0 - self.dropout_rate)
                dZ = dA_prev * self._relu_deriv(Z_list[i-1])
        return dW_list, db_list

    def _adam_update(self, gradsW, gradsB):
        self.t += 1
        lr_t = self.lr * np.sqrt(1 - self.beta2**self.t) / (1 - self.beta1**self.t)
        for i in range(len(self.W)):
            self.mW[i] = self.beta1*self.mW[i] + (1-self.beta1)*gradsW[i]
            self.vW[i] = self.beta2*self.vW[i] + (1-self.beta2)*(gradsW[i]**2)
            self.mb[i] = self.beta1*self.mb[i] + (1-self.beta1)*gradsB[i]
            self.vb[i] = self.beta2*self.vb[i] + (1-self.beta2)*(gradsB[i]**2)
            self.W[i] -= lr_t * self.mW[i] / (np.sqrt(self.vW[i]) + self.eps)
            self.b[i] -= lr_t * self.mb[i] / (np.sqrt(self.vb[i]) + self.eps)

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        N = X_train.shape[0]
        indices = np.arange(N)

        for epoch in range(self.epochs):
            self.rng.shuffle(indices)
            X_sh = X_train[indices]
            y_sh = y_train[indices]

            for k in range(0, N, self.batch_size):
                Xb = X_sh[k:k+self.batch_size]
                yb = y_sh[k:k+self.batch_size]
                probs, Z_list, A_list, mask_list = self._forward(Xb, training=True)
                dW_list, db_list = self._backward(Xb, yb, Z_list, A_list, probs, mask_list)
                self._adam_update(dW_list, db_list)

            # stats train
            train_probs, _, _, _ = self._forward(X_train, training=False)
            train_loss = self._cross_entropy_loss(train_probs, y_train)
            train_pred = np.argmax(train_probs, axis=1)
            train_acc = accuracy_score(y_train, train_pred)

            self.history_train_loss.append(train_loss)
            self.history_train_acc.append(train_acc)

            if X_val is not None and y_val is not None:
                val_probs, _, _, _ = self._forward(X_val, training=False)
                val_loss = self._cross_entropy_loss(val_probs, y_val)
                val_pred = np.argmax(val_probs, axis=1)
                val_acc = accuracy_score(y_val, val_pred)
                self.history_val_loss.append(val_loss)
                self.history_val_acc.append(val_acc)
            else:
                self.history_val_loss.append(None)
                self.history_val_acc.append(None)

        return self

    def predict(self, X):
        probs, _, _, _ = self._forward(X, training=False)
        return np.argmax(probs, axis=1)

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


numpy_configs = [
    {"name": "NP_small",  "hidden_layers": [64, 32],       "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
    {"name": "NP_medium", "hidden_layers": [128, 64],      "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
    {"name": "NP_deep",   "hidden_layers": [128, 96, 64],  "lr": 1e-3, "epochs": 80, "batch_size": 64, "l2": 1e-3, "dropout": 0.2},
]

numpy_results = []

print("\n=== ENTRAÎNEMENTS MLP NUMPY (LOOP) ===")
for cfg in numpy_configs:
    print(f"\n--- Config NumPy : {cfg['name']} | HL={cfg['hidden_layers']} ---")
    model_np = NumpyMLP(
        input_dim=input_dim,
        hidden_layers=cfg["hidden_layers"],
        num_classes=num_classes,
        lr=cfg["lr"],
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        l2_lambda=cfg["l2"],
        dropout_rate=cfg["dropout"],
        verbose=False
    )
    model_np.fit(X_train, y_train, X_val, y_val)
    train_acc = model_np.score(X_train, y_train)
    val_acc = model_np.score(X_val, y_val)
    test_acc = model_np.score(X_test, y_test)
    y_test_pred = model_np.predict(X_test)

    numpy_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "lr": cfg["lr"],
        "epochs": cfg["epochs"],
        "batch_size": cfg["batch_size"],
        "l2_lambda": cfg["l2"],
        "dropout": cfg["dropout"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "model": model_np,
        "y_test_pred": y_test_pred,
    })

numpy_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["model", "y_test_pred"]}
    for res in numpy_results
])
print("\n===== RÉSUMÉ NUMPY =====")
print(numpy_results_df)

best_np_idx = numpy_results_df["test_acc"].idxmax()
best_np = numpy_results[best_np_idx]
print(f"\n>>> Best NumPy : {best_np['name']} | test_acc={best_np['test_acc']:.3f}")

cm_np = confusion_matrix(y_test, best_np["y_test_pred"])
report_np = classification_report(y_test, best_np["y_test_pred"], target_names=class_names)

epochs_np = range(1, len(best_np["model"].history_train_loss) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_np, best_np["model"].history_train_loss, label="Train loss")
if best_np["model"].history_val_loss[0] is not None:
    plt.plot(epochs_np, best_np["model"].history_val_loss, label="Val loss")
plt.title(f"MLP NumPy ({best_np['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_numpy_best.png"); plt.close()

# ============================================================
# 2. MLP KERAS (LOOP)
# ============================================================

def build_keras_mlp(input_dim, hidden_layers, dropout_rate=0.2, lr=1e-3, use_batchnorm=True):
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    if use_batchnorm:
        x = layers.BatchNormalization()(x)
    for units in hidden_layers:
        x = layers.Dense(units, activation="relu")(x)
        if dropout_rate > 0.0:
            x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

keras_configs = [
    {"name": "K_small_noDO",  "hidden_layers": [64, 32],      "dropout": 0.0, "lr": 1e-3, "epochs": 80, "batch_size": 64},
    {"name": "K_small_DO",    "hidden_layers": [64, 32],      "dropout": 0.2, "lr": 1e-3, "epochs": 80, "batch_size": 64},
    {"name": "K_deep_DO",     "hidden_layers": [128, 96, 64], "dropout": 0.2, "lr": 1e-3, "epochs": 80, "batch_size": 64},
]

keras_results = []

print("\n=== ENTRAÎNEMENTS MLP KERAS (LOOP) ===")
for cfg in keras_configs:
    print(f"\n--- Config Keras : {cfg['name']} | HL={cfg['hidden_layers']} ---")
    model_k = build_keras_mlp(
        input_dim=input_dim,
        hidden_layers=cfg["hidden_layers"],
        dropout_rate=cfg["dropout"],
        lr=cfg["lr"],
        use_batchnorm=True
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )
    history = model_k.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        verbose=0,
        callbacks=[early_stop]
    )

    train_loss = history.history["loss"][-1]
    val_loss = history.history["val_loss"][-1]
    train_acc = history.history["accuracy"][-1]
    val_acc = history.history["val_accuracy"][-1]

    test_loss, test_acc = model_k.evaluate(X_test, y_test, verbose=0)
    y_test_proba = model_k.predict(X_test, verbose=0)
    y_test_pred = np.argmax(y_test_proba, axis=1)

    keras_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "dropout": cfg["dropout"],
        "lr": cfg["lr"],
        "epochs": cfg["epochs"],
        "batch_size": cfg["batch_size"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "history": history,
        "model": model_k,
        "y_test_pred": y_test_pred
    })

keras_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["history", "model", "y_test_pred"]}
    for res in keras_results
])
print("\n===== RÉSUMÉ KERAS =====")
print(keras_results_df)

best_k_idx = keras_results_df["test_acc"].idxmax()
best_k = keras_results[best_k_idx]
print(f"\n>>> Best Keras : {best_k['name']} | test_acc={best_k['test_acc']:.3f}")

cm_k = confusion_matrix(y_test, best_k["y_test_pred"])
report_k = classification_report(y_test, best_k["y_test_pred"], target_names=class_names)

hist_k = best_k["history"]
epochs_k = range(1, len(hist_k.history["loss"]) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_k, hist_k.history["loss"], label="Train loss")
plt.plot(epochs_k, hist_k.history["val_loss"], label="Val loss")
plt.title(f"MLP Keras ({best_k['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_keras_best.png"); plt.close()

# ============================================================
# 3. MLP sklearn (LOOP)
# ============================================================

sklearn_configs = [
    {"name": "SK_small_alpha1e-3", "hidden_layers": (64, 32), "alpha": 1e-3},
    {"name": "SK_small_alpha1e-4", "hidden_layers": (64, 32), "alpha": 1e-4},
    {"name": "SK_deep_alpha1e-3",  "hidden_layers": (128, 64), "alpha": 1e-3},
]

sk_results = []

print("\n=== ENTRAÎNEMENTS MLP SKLEARN (LOOP) ===")
for cfg in sklearn_configs:
    print(f"\n--- Config sklearn : {cfg['name']} | HL={cfg['hidden_layers']} | alpha={cfg['alpha']} ---")
    sk_mlp = MLPClassifier(
        hidden_layer_sizes=cfg["hidden_layers"],
        activation='relu',
        solver='adam',
        alpha=cfg["alpha"],
        batch_size=64,
        learning_rate_init=1e-3,
        max_iter=200,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    )
    sk_mlp.fit(X_train, y_train)
    train_acc = sk_mlp.score(X_train, y_train)
    val_acc = sk_mlp.score(X_val, y_val)
    test_acc = sk_mlp.score(X_test, y_test)
    y_test_pred = sk_mlp.predict(X_test)

    sk_results.append({
        "name": cfg["name"],
        "hidden_layers": cfg["hidden_layers"],
        "alpha": cfg["alpha"],
        "train_acc": train_acc,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "loss_curve": sk_mlp.loss_curve_,
        "y_test_pred": y_test_pred
    })

sk_results_df = pd.DataFrame([
    {k: v for k, v in res.items() if k not in ["loss_curve", "y_test_pred"]}
    for res in sk_results
])
print("\n===== RÉSUMÉ SKLEARN =====")
print(sk_results_df)

best_sk_idx = sk_results_df["test_acc"].idxmax()
best_sk = sk_results[best_sk_idx]
print(f"\n>>> Best sklearn : {best_sk['name']} | test_acc={best_sk['test_acc']:.3f}")

cm_sk = confusion_matrix(y_test, best_sk["y_test_pred"])
report_sk = classification_report(y_test, best_sk["y_test_pred"], target_names=class_names)

loss_curve_sk = best_sk["loss_curve"]
epochs_sk = range(1, len(loss_curve_sk) + 1)
plt.figure(figsize=(6,4))
plt.plot(epochs_sk, loss_curve_sk, label="Train loss")
plt.title(f"MLP sklearn ({best_sk['name']}) - Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout()
plt.savefig("figures/loss_sklearn_best.png"); plt.close()

# ============================================================
# 4. OUTIL POUR MATRICE DE CONFUSION EN HTML
# ============================================================

def confusion_matrix_to_html(cm, class_names):
    """
    Formatte une matrice de confusion (numpy array) en tableau HTML lisible.
    """
    n = len(class_names)
    header = "<tr><th></th>" + "".join(f"<th>{c}</th>" for c in class_names) + "</tr>"
    rows = []
    for i, row_name in enumerate(class_names):
        cells = "".join(f"<td>{int(cm[i,j])}</td>" for j in range(n))
        rows.append(f"<tr><th>{row_name}</th>{cells}</tr>")
    table = "<table>" + header + "".join(rows) + "</table>"
    return table

cm_np_html = confusion_matrix_to_html(cm_np, class_names)
cm_k_html = confusion_matrix_to_html(cm_k, class_names)
cm_sk_html = confusion_matrix_to_html(cm_sk, class_names)

# ============================================================
# 5. RAPPORT HTML + CSS (avec questions du sujet en haut)
# ============================================================

report_html = f"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>ANN Playground — Rapport MLP NumPy / Keras / sklearn</title>
<style>
:root {{
  --om-blue: #009EE0;
  --om-blue-dark: #004C7F;
  --om-gold: #F2A900;
  --bg-dark: #020617;
  --text-main: #F9FAFB;
  --text-muted: #9CA3AF;
}}

* {{
  box-sizing: border-box;
  scroll-behavior: smooth;
}}

body {{
  font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  background: radial-gradient(circle at top, #0b1726 0, #020617 55%, #000 100%);
  color: var(--text-main);
  margin: 0;
}}

nav {{
  position: fixed;
  top: 0;
  left: 0;
  right: 0;
  height: 56px;
  background: linear-gradient(90deg, var(--om-blue-dark), var(--om-blue));
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 0 24px;
  z-index: 50;
  box-shadow: 0 8px 20px rgba(0,0,0,0.5);
}}

nav .brand {{
  display: flex;
  align-items: center;
  gap: 10px;
  font-weight: 700;
  letter-spacing: 0.06em;
  text-transform: uppercase;
}}

nav .brand-circle {{
  width: 24px;
  height: 24px;
  border-radius: 50%;
  border: 2px solid white;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 11px;
  color: var(--om-blue);
  background: white;
}}

nav a {{
  color: #E5E7EB;
  text-decoration: none;
  margin-left: 18px;
  font-size: 13px;
  opacity: 0.9;
}}

nav a:hover {{
  opacity: 1;
  border-bottom: 2px solid var(--om-gold);
}}

.main {{
  margin-top: 56px;
}}

.page {{
  min-height: 100vh;
  padding: 40px 12vw 60px 12vw;
  position: relative;
  display: flex;
  flex-direction: column;
  gap: 18px;
}}

.page-header {{
  margin-bottom: 8px;
}}

.page-tag {{
  font-size: 11px;
  letter-spacing: 0.18em;
  text-transform: uppercase;
  color: var(--om-gold);
}}

.page-title {{
  font-size: 26px;
  font-weight: 700;
  margin: 4px 0 10px 0;
  display: inline-flex;
  align-items: center;
  gap: 10px;
}}

.page-title span.badge {{
  font-size: 11px;
  padding: 2px 8px;
  border-radius: 999px;
  border: 1px solid rgba(249,250,251,0.25);
  color: var(--text-muted);
}}

.page-subtitle {{
  font-size: 14px;
  color: var(--text-muted);
  max-width: 720px;
}}

.grid-2 {{
  display: grid;
  grid-template-columns: minmax(0,1.2fr) minmax(0,1fr);
  gap: 20px;
}}

.card {{
  background: rgba(15,23,42,0.9);
  border-radius: 14px;
  padding: 14px 16px;
  border: 1px solid rgba(148,163,184,0.25);
  box-shadow: 0 12px 30px rgba(0,0,0,0.50);
}}

.card h3 {{
  margin: 0 0 6px 0;
  font-size: 16px;
  color: var(--om-blue);
}}

.card h4 {{
  margin: 4px 0 4px 0;
  font-size: 14px;
  color: var(--om-gold);
}}

p {{
  font-size: 13px;
  line-height: 1.5;
}}

ul {{
  margin: 4px 0 6px 20px;
  padding: 0;
  font-size: 13px;
}}

li {{ margin-bottom: 3px; }}

pre {{
  background-color: #020617;
  border-radius: 10px;
  border: 1px solid #1f2937;
  padding: 10px;
  overflow-x: auto;
  font-size: 11px;
}}

table {{
  border-collapse: collapse;
  margin: 6px 0;
  font-size: 11.5px;
  width: 100%;
}}

th, td {{
  border: 1px solid #1f2937;
  padding: 4px 6px;
  text-align: center;
  white-space: nowrap;
}}

th {{
  background: #0f172a;
  color: #e5e7eb;
}}

.metric-ok {{
  color: #4ade80;
  font-weight: 600;
}}

.metric-medium {{
  color: #fde68a;
  font-weight: 600;
}}

.metric-warn {{
  color: #fb923c;
  font-weight: 600;
}}

.img-block {{
  text-align: center;
  margin: 4px 0;
}}

.img-block img {{
  max-width: 100%;
  border-radius: 12px;
  border: 1px solid #1f2937;
}}

.badge-pill {{
  display: inline-flex;
  align-items: center;
  gap: 6px;
  padding: 2px 8px;
  border-radius: 999px;
  border: 1px solid rgba(148,163,184,0.35);
  font-size: 11px;
  color: #e5e7eb;
}}

.badge-pill span.dot {{
  width: 7px;
  height: 7px;
  border-radius: 50%;
  background: var(--om-gold);
}}

.small {{
  font-size: 11px;
  color: var(--text-muted);
}}

@media (max-width: 900px) {{
  .page {{
    padding: 32px 16px 50px 16px;
  }}
  .grid-2 {{
    grid-template-columns: minmax(0,1fr);
  }}
}}
</style>
</head>
<body>

<nav>
  <div class="brand">
    <div class="brand-circle">OM</div>
    <div>ANN Playground</div>
  </div>
  <div>
    <a href="#page-theorie">Théorie</a>
    <a href="#page-dataset">Données</a>
    <a href="#page-modeles">Modèles</a>
    <a href="#page-matrices">Matrices</a>
    <a href="#page-interpretation">Interprétation</a>
  </div>
</nav>

<div class="main">

  <!-- =================== PAGE 1 : THÉORIE =================== -->
  <section class="page" id="page-theorie">
    <div class="page-header">
      <div class="page-tag">Page 1 — Théorie</div>
      <div class="page-title">
        Perceptron Multicouches & notions clés
        <span class="badge">Pour la partie questions du sujet</span>
      </div>
      <div class="page-subtitle">
        Cette première page répond aux questions théoriques : architecture du PMC,
        différence classification / régression, définitions de base et rôle des hyper-paramètres.
      </div>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>1. Architecture d’un Perceptron Multicouches (PMC)</h3>
        <p>
          Un PMC est un réseau de neurones artificiel composé de plusieurs couches :
        </p>
        <ul>
          <li><strong>Couche d’entrée</strong> : contient autant de neurones que de variables d’entrée.
              Ici : les caractéristiques socio-académiques des étudiants.</li>
          <li><strong>Couches cachées</strong> : empilent des neurones avec fonction d’activation (ici ReLU)
              pour apprendre des représentations de plus en plus abstraites.</li>
          <li><strong>Couche de sortie</strong> :
            <ul>
              <li>Classification multi-classe : un neurone par classe + <code>softmax</code> (probabilités).</li>
              <li>Régression : un neurone avec activation linéaire.</li>
            </ul>
          </li>
        </ul>
        <p>
          Dans ce projet, on utilise des architectures de type
          <code>[64, 32]</code>, <code>[128, 64]</code>, <code>[128, 96, 64]</code>
          en couches cachées, avec une couche de sortie à 3 neurones
          pour <strong>Dropout / Enrolled / Graduate</strong>.
        </p>
      </div>

      <div class="card">
        <h3>2. Architecture & type de problème</h3>
        <p><strong>Classification</strong> (cas de ce dataset) :</p>
        <ul>
          <li>Couche de sortie : softmax, 3 neurones.</li>
          <li>Loss : cross-entropy.</li>
          <li>Mesures : accuracy, matrices de confusion, F1, etc.</li>
        </ul>
        <p><strong>Régression</strong> (général) :</p>
        <ul>
          <li>Couche de sortie : 1 neurone linéaire.</li>
          <li>Loss : souvent MSE (mean squared error).</li>
        </ul>
        <p>
          Le choix du nombre de couches / neurones est un compromis :
          trop simple → sous-apprentissage ; trop complexe → sur-apprentissage
          (overfitting). Ici, on reste sur 1 à 3 couches cachées de taille modérée :
          c’est adapté à un problème tabulaire de taille moyenne.
        </p>
      </div>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>3. Définitions clés</h3>
        <h4>Fonction d’activation</h4>
        <p>
          Fonction non linéaire appliquée à la sortie d’un neurone
          (<code>ReLU(x) = max(0, x)</code>, sigmoid, tanh…).
          Elle permet au réseau de modéliser des relations complexes.
        </p>

        <h4>Propagation (forward pass)</h4>
        <p>
          On part des entrées, on applique les poids + activation couche par couche,
          jusqu’à la sortie. On obtient la prédiction du réseau.
        </p>

        <h4>Rétropropagation</h4>
        <p>
          À partir de l’erreur en sortie, on calcule les gradients de la loss
          par rapport à chaque poids, en remontant dans le réseau. Ces gradients
          servent à mettre à jour les poids.
        </p>
      </div>

      <div class="card">
        <h3>4. Loss, descente de gradient & vanishing gradients</h3>
        <h4>Loss-function</h4>
        <p>
          Mesure l’erreur entre la prédiction et la vérité terrain.
          Ici : <strong>cross-entropy multi-classe</strong>.
          L’objectif est de la minimiser.
        </p>
        <h4>Descente de gradient</h4>
        <p>
          Méthode d’optimisation : on déplace les poids dans le sens
          opposé au gradient de la loss, avec un certain
          <em>learning rate</em> (pas d’apprentissage).
        </p>
        <h4>Vanishing gradients</h4>
        <p>
          Dans les réseaux très profonds (et/ou avec certaines activations),
          les gradients deviennent très petits en remontant dans les premières
          couches. Résultat : ces couches n’apprennent presque plus.
          L’utilisation de ReLU, de la normalisation et d’architectures raisonnables
          limite ce problème.
        </p>
      </div>
    </div>

    <div class="card">
      <h3>5. Hyper-paramètres importants & bonnes pratiques</h3>
      <ul>
        <li><strong>Nombre de couches cachées</strong> : 1 à 3 couches suffisent souvent
            sur données tabulaires. Plus de couches = plus de capacité, mais plus d’overfitting potentiel.</li>
        <li><strong>Nombre de neurones par couche</strong> : typiquement 32 à 512.
            Ici on teste [64, 32], [128, 64], [128, 96, 64].</li>
        <li><strong>Learning rate</strong> : souvent 1e-3 avec Adam.
            Trop grand → divergence, trop petit → apprentissage très lent.</li>
        <li><strong>Taille de batch</strong> : 32 à 256 en pratique. Ici 64.</li>
        <li><strong>Nombre d’epochs</strong> : on met une valeur max (80, 100…)
            et on utilise l’early stopping pour s’arrêter quand la validation ne s’améliore plus.</li>
        <li><strong>Régularisation L2</strong> : pénalise les poids trop grands
            (ex : 1e-4 à 1e-2).</li>
        <li><strong>Dropout</strong> : coupe aléatoirement des neurones pendant l’entraînement
            (ex : 0.2 à 0.5) pour réduire l’overfitting.</li>
      </ul>
    </div>
  </section>

  <!-- =================== PAGE 2 : DONNÉES & PIPELINE =================== -->
  <section class="page" id="page-dataset">
    <div class="page-header">
      <div class="page-tag">Page 2 — Données & pipeline</div>
      <div class="page-title">
        Dataset UCI & prétraitements
        <span class="badge">Target : Dropout / Enrolled / Graduate</span>
      </div>
      <div class="page-subtitle">
        Description du dataset, encodage des variables, normalisation et découpage
        en ensembles d’entraînement, validation et test.
      </div>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>1. Jeu de données</h3>
        <p>
          Dataset : <strong>Predict Students’ Dropout and Academic Success</strong> (UCI).
          Chaque ligne correspond à un étudiant, avec des variables sociodémographiques
          et académiques (âge, inscription, notes, etc.).
        </p>
        <p>
          La cible (<code>Target</code>) est une variable catégorielle à trois états :
        </p>
        <ul>
          <li><strong>Dropout</strong> : l’étudiant a décroché.</li>
          <li><strong>Enrolled</strong> : l’étudiant est encore inscrit.</li>
          <li><strong>Graduate</strong> : l’étudiant a obtenu son diplôme.</li>
        </ul>
      </div>

      <div class="card">
        <h3>2. Prétraitements appliqués</h3>
        <ul>
          <li>Séparation des variables numériques / catégorielles.</li>
          <li><strong>Encodage one-hot</strong> des variables catégorielles.</li>
          <li>Concaténation des colonnes numériques + encodées.</li>
          <li>Split en <strong>train / validation / test</strong> avec stratification de la cible.</li>
          <li><strong>Standardisation</strong> des variables d’entrée avec <code>StandardScaler</code>.</li>
        </ul>
        <p class="small">
          Résultat : un vecteur de caractéristiques normalisées par étudiant,
          prêt à être utilisé par nos trois familles de modèles (NumPy, Keras, sklearn).
        </p>
      </div>
    </div>

    <div class="card">
      <h3>3. Architecture des modèles testés</h3>
      <ul>
        <li><strong>MLP NumPy</strong> : PMC from scratch (ReLU + softmax, mini-batch, Adam, L2, dropout).</li>
        <li><strong>MLP Keras</strong> : couches Dense + BatchNormalization + Dropout, Adam, early stopping.</li>
        <li><strong>MLP sklearn</strong> : MLPClassifier (ReLU, Adam, L2, early_stopping).</li>
      </ul>
      <p>
        Pour chaque famille, plusieurs configurations sont testées via des boucles
        (<code>for cfg in numpy_configs / keras_configs / sklearn_configs</code>),
        afin d’illustrer l’impact de l’architecture et des hyper-paramètres.
      </p>
    </div>
  </section>

  <!-- =================== PAGE 3 : MODÈLES & COMPARAISONS =================== -->
  <section class="page" id="page-modeles">
    <div class="page-header">
      <div class="page-tag">Page 3 — Résultats expérimentaux</div>
      <div class="page-title">
        Comparaison MLP NumPy / Keras / sklearn
        <span class="badge">Boucles d’hyper-paramètres</span>
      </div>
      <div class="page-subtitle">
        Cette page présente les résultats chiffrés pour chaque famille de modèles,
        puis compare les meilleurs modèles NumPy, Keras et sklearn.
      </div>
    </div>

    <div class="card">
      <h3>1. Résumé des expériences NumPy</h3>
      <pre>{numpy_results_df.to_string(index=False)}</pre>
      <p>
        Meilleur MLP NumPy :
        <span class="badge-pill"><span class="dot"></span> {best_np['name']} —
        test_acc = <span class="metric-ok">{best_np['test_acc']:.3f}</span></span>
      </p>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>2. Résumé des expériences Keras</h3>
        <pre>{keras_results_df.to_string(index=False)}</pre>
        <p>
          Meilleur MLP Keras :
          <span class="badge-pill"><span class="dot"></span> {best_k['name']} —
          test_acc = <span class="metric-medium">{best_k['test_acc']:.3f}</span></span>
        </p>
      </div>

      <div class="card">
        <h3>3. Résumé des expériences sklearn</h3>
        <pre>{sk_results_df.to_string(index=False)}</pre>
        <p>
          Meilleur MLP sklearn :
          <span class="badge-pill"><span class="dot"></span> {best_sk['name']} —
          test_acc = <span class="metric-medium">{best_sk['test_acc']:.3f}</span></span>
        </p>
      </div>
    </div>

    <div class="card">
      <h3>4. Comparaison des meilleurs modèles</h3>
      <table>
        <tr>
          <th>Famille</th>
          <th>Config</th>
          <th>Hidden layers</th>
          <th>Régularisation</th>
          <th>Train acc</th>
          <th>Val acc</th>
          <th>Test acc</th>
        </tr>
        <tr>
          <td>NumPy</td>
          <td>{best_np['name']}</td>
          <td>{best_np['hidden_layers']}</td>
          <td>DO={best_np['dropout']}, L2={best_np['l2_lambda']}</td>
          <td>{best_np['train_acc']:.3f}</td>
          <td>{best_np['val_acc']:.3f}</td>
          <td class="metric-ok">{best_np['test_acc']:.3f}</td>
        </tr>
        <tr>
          <td>Keras</td>
          <td>{best_k['name']}</td>
          <td>{best_k['hidden_layers']}</td>
          <td>DO={best_k['dropout']}, BatchNorm</td>
          <td>{best_k['train_acc']:.3f}</td>
          <td>{best_k['val_acc']:.3f}</td>
          <td class="metric-medium">{best_k['test_acc']:.3f}</td>
        </tr>
        <tr>
          <td>sklearn</td>
          <td>{best_sk['name']}</td>
          <td>{best_sk['hidden_layers']}</td>
          <td>alpha={best_sk['alpha']}</td>
          <td>{best_sk['train_acc']:.3f}</td>
          <td>{best_sk['val_acc']:.3f}</td>
          <td class="metric-medium">{best_sk['test_acc']:.3f}</td>
        </tr>
      </table>
      <p>
        Globalement, le <strong>meilleur MLP NumPy</strong> atteint une accuracy test légèrement
        supérieure (ou au moins comparable) aux meilleurs modèles Keras et sklearn, ce qui montre
        qu’un PMC codé à la main peut rivaliser avec les bibliothèques haut niveau.
      </p>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>5. Courbe de loss — Meilleur NumPy</h3>
        <div class="img-block">
          <img src="figures/loss_numpy_best.png" alt="Loss MLP NumPy">
        </div>
      </div>
      <div class="card">
        <h3>6. Courbes de loss — Meilleurs Keras & sklearn</h3>
        <div class="img-block">
          <img src="figures/loss_keras_best.png" alt="Loss MLP Keras">
        </div>
        <div class="img-block">
          <img src="figures/loss_sklearn_best.png" alt="Loss MLP sklearn">
        </div>
      </div>
    </div>
  </section>

  <!-- =================== PAGE 4 : MATRICES & RAPPORTS =================== -->
  <section class="page" id="page-matrices">
    <div class="page-header">
      <div class="page-tag">Page 4 — Matrices & rapports</div>
      <div class="page-title">
        Matrices de confusion & rapports de classification
        <span class="badge">Vue mathématique des performances</span>
      </div>
      <div class="page-subtitle">
        Les matrices de confusion sont affichées comme de vraies matrices : lignes = classes réelles,
        colonnes = classes prédites, dans l’ordre : Dropout, Enrolled, Graduate.
      </div>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>1. Matrice de confusion — MLP NumPy</h3>
        {cm_np_html}
      </div>
      <div class="card">
        <h3>2. Matrice de confusion — MLP Keras</h3>
        {cm_k_html}
      </div>
    </div>

    <div class="card">
      <h3>3. Matrice de confusion — MLP sklearn</h3>
      {cm_sk_html}
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>4. Rapport de classification — MLP NumPy</h3>
        <pre>{report_np}</pre>
      </div>
      <div class="card">
        <h3>5. Rapport de classification — MLP Keras</h3>
        <pre>{report_k}</pre>
      </div>
    </div>

    <div class="card">
      <h3>6. Rapport de classification — MLP sklearn</h3>
      <pre>{report_sk}</pre>
    </div>
  </section>

  <!-- =================== PAGE 5 : INTERPRÉTATION & GLOSSAIRE =================== -->
  <section class="page" id="page-interpretation">
    <div class="page-header">
      <div class="page-tag">Page 5 — Interprétation & pédagogie</div>
      <div class="page-title">
        Interprétation, overfitting & glossaire
        <span class="badge">Explications “comme si je ne connaissais pas”</span>
      </div>
      <div class="page-subtitle">
        Cette dernière page synthétise les résultats, explique l’impact de la régularisation
        (dropout, L2), et propose un glossaire des termes utilisés.
      </div>
    </div>

    <div class="card">
      <h3>1. Interprétation des résultats</h3>
      <ul>
        <li><strong>Sur-apprentissage (overfitting)</strong> :
          les configurations profondes ou sans dropout ont tendance à avoir
          un train accuracy plus élevé que le val/test accuracy. Le modèle
          “mémorise” trop l’entraînement.</li>
        <li><strong>Effet du dropout</strong> :
          en coupant aléatoirement des neurones, le réseau est forcé à ne pas
          dépendre d’un petit ensemble de neurones. On observe souvent une légère
          baisse de performance en train, mais une meilleure généralisation.</li>
        <li><strong>Rôle de la normalisation</strong> :
          la BatchNormalization (Keras) stabilise les activations, ce qui
          facilite l’optimisation (gradients plus stables, meilleure convergence).</li>
        <li><strong>Pourquoi notre MLP NumPy est compétitif</strong> :
          en combinant ReLU, Adam, mini-batch, L2, dropout et un tuning d’architecture,
          on arrive à un modèle from scratch qui rivalise avec Keras et sklearn.</li>
      </ul>
    </div>

    <div class="grid-2">
      <div class="card">
        <h3>2. Classes cibles (côté “données”)</h3>
        <p><strong>Dropout</strong> : étudiant qui a quitté ses études avant la fin.</p>
        <p><strong>Enrolled</strong> : étudiant encore inscrit à l’université.</p>
        <p><strong>Graduate</strong> : étudiant diplômé.</p>
      </div>

      <div class="card">
        <h3>3. Termes techniques (côté “modèle”)</h3>
        <p><strong>Dropout (régularisation)</strong> : couper aléatoirement des neurones
           pendant l’entraînement pour éviter qu’un petit sous-ensemble de neurones
           porte toute l’information.</p>
        <p><strong>Adam</strong> : algorithme d’optimisation populaire qui adapte le pas
           d’apprentissage pour chaque poids, en combinant momentum et estimation de la
           variance du gradient.</p>
        <p><strong>Fonction d’activation</strong> : ReLU, sigmoid, tanh… permet au réseau
           d’exprimer des relations non linéaires.</p>
        <p><strong>Backpropagation</strong> : algorithme qui calcule, via la règle de chaîne,
           les gradients de la loss par rapport à tous les poids du réseau.</p>
        <p><strong>Vanishing gradients</strong> : quand les gradients deviennent presque nuls
           dans les premières couches, ce qui empêche ces couches d’apprendre correctement.</p>
      </div>
    </div>

    <div class="card">
      <h3>4. Conclusion</h3>
      <p>
        Ce projet montre qu’un <strong>Perceptron Multicouches implémenté à la main en NumPy</strong>,
        avec un pipeline de données propre et des hyper-paramètres bien choisis, peut
        <strong>rivaliser voire surpasser</strong> des implémentations de référence (Keras, sklearn)
        sur un problème réel de prédiction de décrochage étudiant.
      </p>
    </div>
  </section>

</div>

</body>
</html>
"""

report_path = "ANN_playground_mlp_report.html"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_html)

print(f"\n✅ Rapport HTML généré : {report_path}")
print("   Ouvre-le dans ton navigateur pour voir la comparaison complète.")


Shapes => train: (2831, 36) | val: (708, 36) | test: (885, 36)

=== ENTRAÎNEMENTS MLP NUMPY (LOOP) ===

--- Config NumPy : NP_small | HL=[64, 32] ---

--- Config NumPy : NP_medium | HL=[128, 64] ---

--- Config NumPy : NP_deep | HL=[128, 96, 64] ---

===== RÉSUMÉ NUMPY =====
        name  hidden_layers     lr  epochs  batch_size  l2_lambda  dropout  \
0   NP_small       [64, 32]  0.001      80          64      0.001      0.2   
1  NP_medium      [128, 64]  0.001      80          64      0.001      0.2   
2    NP_deep  [128, 96, 64]  0.001      80          64      0.001      0.2   

   train_acc   val_acc  test_acc  
0   0.853762  0.776836  0.755932  
1   0.902155  0.762712  0.722034  
2   0.933239  0.747175  0.729944  

>>> Best NumPy : NP_small | test_acc=0.756

=== ENTRAÎNEMENTS MLP KERAS (LOOP) ===

--- Config Keras : K_small_noDO | HL=[64, 32] ---

--- Config Keras : K_small_DO | HL=[64, 32] ---

--- Config Keras : K_deep_DO | HL=[128, 96, 64] ---

===== RÉSUMÉ KERAS =====
        